In [1]:
#1.Load original dataset

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
candidates = [Path('wine_raw.csv'), Path('../01_Data_Collection/wine_raw.csv'), Path('01_Data_Collection/wine_raw.csv'), Path('Wine_ML_Assignment/01_Data_Collection/wine_raw.csv')]
path = next((p for p in candidates if p.exists()), None)
if path is not None:
    df = pd.read_csv(path)
else:
    from sklearn.datasets import load_wine
    wine = load_wine(as_frame=True)
    df = wine.frame.rename(columns={'target':'class'})
print('Dataset:', path if path else 'scikit-learn UCI Wine fallback')
print('Shape:',df.shape)
display(df.head())

Dataset: wine_raw.csv
Shape: (178, 14)


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,class
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


In [3]:
#2.Data quality checks

In [4]:
print('Missing values:',df.isna().sum().sum())
print('Duplicate rows:',df.duplicated().sum())
print('Target classes:',sorted(df['class'].unique()))
X = df.drop(columns=['class'])
y = df['class'].copy()
print('X shape:',X.shape,'y shape:',y.shape)

Missing values: 0
Duplicate rows: 0
Target classes: [np.int64(0), np.int64(1), np.int64(2)]
X shape: (178, 13) y shape: (178,)


In [5]:
#3.Train/test split before fitting preprocessing

In [6]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
print('Train:',X_train.shape,'Test:',X_test.shape)

Train: (142, 13) Test: (36, 13)


In [7]:
# Method A preprocess input x only

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
x_processor = Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
X_train_A = x_processor.fit_transform(X_train)
X_test_A = x_processor.transform(X_test)
y_train_A,y_test_A=y_train.copy(),y_test.copy()
print('X processed:',X_train_A.shape,'Target values:',sorted(y_train_A.unique()))

X processed: (142, 13) Target values: [np.int64(0), np.int64(1), np.int64(2)]


In [9]:
#Method B — Preprocess input X and output y

In [10]:
from sklearn.preprocessing import LabelEncoder
encoder=LabelEncoder()
X_train_B = x_processor.fit_transform(X_train)
X_test_B = x_processor.transform(X_test)
y_train_B = encoder.fit_transform(y_train)
y_test_B = encoder.transform(y_test)
print('Original classes:',list(encoder.classes_))
print('Encoded classes:',sorted(np.unique(y_train_B)))
print('Target unchanged?',np.array_equal(y_train_A.to_numpy(),y_train_B))
print('Inputs identical?',np.allclose(X_train_A,X_train_B))

Original classes: [np.int64(0), np.int64(1), np.int64(2)]
Encoded classes: [np.int64(0), np.int64(1), np.int64(2)]
Target unchanged? True
Inputs identical? True
